In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV, learning_curve
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.decomposition import PCA
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR

In [ ]:
# 1. Caricamento del dataset
diabetes = load_diabetes()
X = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)
y = pd.Series(diabetes.target, name='target')

In [ ]:
# Unione in un unico DataFrame per l'analisi esplorativa
df = X.copy()
df['target'] = y

# Analisi descrittiva iniziale
print("--- STATISTICHE DESCRITTIVE ---")
print(df.describe())

In [ ]:
# Set dello stile dei grafici
sns.set_theme(style="whitegrid")

# Istogrammi delle feature principali e del target
df[['age', 'bmi', 'bp', 'target']].hist(bins=20, figsize=(10, 8))
plt.suptitle("Distribuzione di alcune Feature e del Target")
plt.show()

In [ ]:

# Boxplot per identificare potenziali outlier
plt.figure(figsize=(12, 6))
sns.boxplot(data=X)
plt.title("Boxplot delle Feature (Identificazione Outlier)")
plt.show()

In [ ]:

# Matrice di Correlazione (in alternativa alla Scatter Matrix per leggibilità)
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="coolwarm", cbar=True)
plt.title("Matrice di Correlazione di Pearson")
plt.show()

In [ ]:

# Separazione delle feature dal target
X_data = df.drop(columns=['target'])
y_data = df['target']

# Split 80% Training e 20% Test
X_train, X_test, y_train, y_test = train_test_split(
    X_data, y_data, test_size=0.2, random_state=42
)

print(f"Dimensioni del Training Set: {X_train.shape}")
print(f"Dimensioni del Test Set: {X_test.shape}")


In [ ]:

# Definizione dei modelli
models = {
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'Lasso': Lasso(alpha=1.0),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'KNN': KNeighborsRegressor(),
    'SVM (SVR)': SVR(kernel='rbf')
}

In [ ]:

# Configurazione della K-Fold Cross-Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)

print("--- CONFRONTO INIZIALE MODELLI (5-Fold CV) ---")
results = {}

In [ ]:

for name, model in models.items():
    # Creazione della pipeline: Scaling + Modello
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('regressor', model)
    ])
    
    # Valutazione basata su NMSE (Negative Mean Squared Error)
    cv_scores = cross_val_score(
        pipeline, X_train, y_train, cv=kf, scoring='neg_mean_squared_error'
    )
     # Convertiamo in valori positivi per facilitare la lettura (MSE medio)
    mse_scores = -cv_scores
    results[name] = mse_scores
    print(f"{name}: MSE Medio = {mse_scores.mean():.2f} (+/- {mse_scores.std():.2f})")

In [ ]:
# Definizione della Pipeline per il modello scelto
ridge_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('ridge', Ridge())
])

# Spazio degli iperparametri da esplorare
param_grid = {
    'ridge': [Ridge()],
    'ridge__alpha': np.logspace(-3, 3, 100) # 100 valori da 0.001 a 1000
}

# Esecuzione della Grid Search
grid_search = GridSearchCV(
    ridge_pipeline, param_grid, cv=kf, scoring='neg_mean_squared_error', n_jobs=-1
)
grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_
print(f"Miglior iperparametro (alpha): {grid_search.best_params_['ridge__alpha']:.4f}")

In [ ]:
# --- VALUTAZIONE FINALE SUL TEST SET ---
y_pred = best_model.predict(X_test)

final_mse = mean_squared_error(y_test, y_pred)
final_r2 = r2_score(y_test, y_pred)

print("\n--- METRICHE SUL TEST SET (Modello Ottimizzato) ---")
print(f"Mean Squared Error (MSE): {final_mse:.2f}")
print(f"Coefficiente di Determinazione (R²): {final_r2:.2f}")

In [ ]:

train_sizes, train_scores, test_scores = learning_curve(
    best_model, X_train, y_train, cv=kf, scoring='neg_mean_squared_error',
    train_sizes=np.linspace(0.1, 1.0, 10), n_jobs=-1
)

# Calcolo delle medie (invertendo il segno per avere l'MSE positivo)
train_errors_mean = -np.mean(train_scores, axis=1)
test_errors_mean = -np.mean(test_scores, axis=1)

In [ ]:
# Plot delle Learning Curves
plt.figure(figsize=(10, 6))
plt.plot(train_sizes, train_errors_mean, 'o-', color="r", label="Training error")
plt.plot(train_sizes, test_errors_mean, 'o-', color="g", label="Cross-validation error")
plt.title("Learning Curves (Modello Ottimizzato)")
plt.xlabel("Dimensione del Training Set")
plt.ylabel("Mean Squared Error (MSE)")
plt.legend(loc="best")
plt.grid(True)
plt.show()

In [ ]:
# 1. Riduzione dimensionale con PCA a 2 componenti
pca_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=2))
])

X_train_pca = pca_pipeline.fit_transform(X_train)
X_test_pca = pca_pipeline.transform(X_test)

In [ ]:
# 2. Addestramento del modello ottimizzato nello spazio bidimensionale della PCA
ridge_pca = Ridge(alpha=grid_search.best_params_['ridge__alpha'])
ridge_pca.fit(X_train_pca, y_train)

# Creazione di una meshgrid per plottare la superficie/piano di regressione
x_min, x_max = X_train_pca[:, 0].min() - 1, X_train_pca[:, 0].max() + 1
y_min, y_max = X_train_pca[:, 1].min() - 1, X_train_pca[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100), np.linspace(y_min, y_max, 100))

# Predizione sui punti della meshgrid
grid_points = np.c_[xx.ravel(), yy.ravel()]
zz = ridge_pca.predict(grid_points).reshape(xx.shape)

In [ ]:
# 3. Plot in 3D del Piano di Regressione
fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')

# Plot della superficie di regressione
surf = ax.plot_surface(xx, yy, zz, cmap='viridis', alpha=0.6, edgecolor='none')

# Scatter plot dei dati di test reali
scat = ax.scatter(X_test_pca[:, 0], X_test_pca[:, 1], y_test, c='red', marker='o', alpha=0.8, label='Dati di Test Reali')

ax.set_title("Piano di Regressione Ridge nello Spazio PCA Bidimensionale")
ax.set_xlabel("Componente Principale 1 (PC1)")
ax.set_ylabel("Componente Principale 2 (PC2)")
ax.set_zlabel("Target (Progressione Diabete)")
ax.legend()
fig.colorbar(surf, ax=ax, shrink=0.5, aspect=5)
plt.show()